IMplementirati BILO KAKAV faktorizacijski model za recommendation NA BILO kojem setu.
Mora se vidjeti skup podataka, mora se videjti SPARSE (suplja) matrica
I kakva je matrica poslije

In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

# Učitavanje dataseta sa Kaggle-a (6810 knjiga)
df = pd.read_csv("books.csv")
df = df.dropna(subset=['average_rating', 'categories', 'authors'])
df = df[df['average_rating'] > 0]

print(f"Učitano knjiga: {len(df)}")
print(f"Kolone: {df.columns.tolist()}")
df[['title', 'authors', 'categories', 'average_rating', 'ratings_count']].head(5)

Učitano knjiga: 6590
Kolone: ['isbn13', 'isbn10', 'title', 'subtitle', 'authors', 'categories', 'thumbnail', 'description', 'published_year', 'average_rating', 'num_pages', 'ratings_count']


,title,authors,categories,average_rating,ratings_count
0,Gilead,Marilynne Robinson,Fiction,3.85,361.0
1,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,3.83,5164.0
2,The One Tree,Stephen R. Donaldson,American fiction,3.97,172.0
3,Rage of angels,Sidney Sheldon,Fiction,3.93,29532.0
4,The Four Loves,Clive Staples Lewis,Christian life,4.15,33684.0


In [2]:
# Gradim author × category matricu
# Vrijednost = prosječni rating knjiga tog autora u toj kategoriji (iz stvarnih podataka)
# NaN = autor nema knjige u toj kategoriji → 0 (prazno/sparse polje)

df['author1'] = df['authors'].str.split(';').str[0].str.strip()
top_authors = df['author1'].value_counts().head(20).index
top_cats    = df['categories'].value_counts().head(15).index
df2   = df[df['author1'].isin(top_authors) & df['categories'].isin(top_cats)]
pivot = df2.pivot_table(index='author1', columns='categories',
                        values='average_rating', aggfunc='mean')

authors = pivot.index.tolist()    
cats = pivot.columns.tolist()  
n_cats = len(cats)              

R = pivot.fillna(0).values
print(f"Matrica: {R.shape[0]} autora × {R.shape[1]} kategorija")
pivot.round(2)

Matrica: 20 autora × 11 kategorija


categories,Biography & Autobiography,Comics & Graphic Novels,Drama,Fiction,History,Juvenile Fiction,Juvenile Nonfiction,Literary Collections,Literary Criticism,Philosophy,Poetry
author1,,,,,,,,,,,
Agatha Christie,NaN,NaN,NaN,3.96,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Charles Dickens,NaN,NaN,NaN,4.00,NaN,4.19,NaN,NaN,NaN,NaN,NaN
Clive Cussler,NaN,NaN,NaN,3.91,3.84,NaN,NaN,NaN,NaN,NaN,NaN
Jane Austen,NaN,NaN,NaN,4.18,NaN,NaN,NaN,NaN,3.92,NaN,NaN
Janet Evanovich,NaN,NaN,NaN,3.91,NaN,NaN,NaN,NaN,NaN,NaN,NaN
John Ronald Reuel Tolkien,4.14,NaN,NaN,4.14,NaN,4.26,NaN,4.09,NaN,NaN,3.96
Margaret Weis,NaN,4.14,NaN,3.78,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Mercedes Lackey,NaN,NaN,NaN,4.01,NaN,NaN,NaN,NaN,4.32,NaN,NaN
Neil Gaiman,NaN,4.13,NaN,4.08,NaN,4.05,NaN,NaN,NaN,NaN,NaN


In [3]:
# Konverzija u CSR sparse format
sparse = csr_matrix(R)
total  = R.shape[0] * R.shape[1]
print(f"Oblik: {sparse.shape}")
print(f"Ukupno: {total} ćelija")
print(f"Nnz: {sparse.nnz} (ima vrijednost)")
print(f"Nule: {total - sparse.nnz} (autor nema knjiga u toj kategoriji)")
print(f"Density: {sparse.nnz/total:.1%}")
print(f"Sparsity: {1 - sparse.nnz/total:.1%}")

Oblik: (20, 11)
Ukupno: 220 ćelija
Nnz: 49 (ima vrijednost)
Nule: 171 (autor nema knjiga u toj kategoriji)
Density: 22.3%
Sparsity: 77.7%


In [4]:
# SPARSE matrica PRIJE faktorizacije
print("Matrica R PRIJE:")
df_R = pd.DataFrame(np.round(R, 2), index=authors, columns=cats)
print(df_R.to_string()) 

Matrica R PRIJE:
                           Biography & Autobiography  Comics & Graphic Novels  Drama  Fiction  History  Juvenile Fiction  Juvenile Nonfiction  Literary Collections  Literary Criticism  Philosophy  Poetry
Agatha Christie                                 0.00                     0.00   0.00     3.96     0.00              0.00                 0.00                  0.00                0.00        0.00    0.00
Charles Dickens                                 0.00                     0.00   0.00     4.00     0.00              4.19                 0.00                  0.00                0.00        0.00    0.00
Clive Cussler                                   0.00                     0.00   0.00     3.91     3.84              0.00                 0.00                  0.00                0.00        0.00    0.00
Jane Austen                                     0.00                     0.00   0.00     4.18     0.00              0.00                 0.00                  0.00    

In [5]:
# Matrix Factorization — SGD
# R ≈ P × Q^T
class MatrixFactorization:
    def __init__(self, k=3, lr=0.005, reg=0.02, epochs=2000):
        self.k, self.lr, self.reg, self.epochs = k, lr, reg, epochs

    def fit(self, R):
        n_u, n_i = R.shape
        self.P = np.random.normal(0, 0.1, (n_u, self.k))
        self.Q = np.random.normal(0, 0.1, (n_i, self.k))
        mask = R > 0
        for _ in range(self.epochs):
            R_hat = self.P @ self.Q.T
            E     = (R - R_hat) * mask
            self.P += self.lr * (E @ self.Q   - self.reg * self.P)
            self.Q += self.lr * (E.T @ self.P - self.reg * self.Q)
        return self

    def predict(self):
        return np.clip(self.P @ self.Q.T, 1, 5)

np.random.seed(42)
mf = MatrixFactorization(k=3, lr=0.005, reg=0.02, epochs=2000)
mf.fit(R)
R_pred = mf.predict()
rmse = np.sqrt(np.mean((R[R > 0] - R_pred[R > 0]) ** 2))
print(f"RMSE na poznatim ćelijama: {rmse:.4f}")

RMSE na poznatim ćelijama: 0.0198


In [6]:
# Popunjena matrica POSLIJE faktorizacije
print("Matrica R_pred POSLIJE:")
df_pred = pd.DataFrame(np.round(R_pred, 2), index=authors, columns=cats)
print(df_pred.to_string())  

Matrica R_pred POSLIJE:
                           Biography & Autobiography  Comics & Graphic Novels  Drama  Fiction  History  Juvenile Fiction  Juvenile Nonfiction  Literary Collections  Literary Criticism  Philosophy  Poetry
Agatha Christie                                 2.91                     3.27   2.89     3.95     2.69              3.97                 2.97                  2.20                3.22        2.30    2.40
Charles Dickens                                 2.52                     3.09   2.81     4.05     2.78              4.13                 2.97                  1.87                2.75        1.75    2.23
Clive Cussler                                   3.38                     2.44   1.60     3.91     3.83              4.00                 1.72                  3.23                2.90        2.80    3.27
Jane Austen                                     3.58                     3.62   3.06     4.17     2.93              4.14                 3.08                  2

In [8]:
# Preporuke: top-2 kategorije za svakog autora gdje nema knjiga
for i, author in enumerate(authors):
    missing = [j for j in range(n_cats) if R[i, j] == 0]
    top2 = sorted(missing, key=lambda j: -R_pred[i, j])[:2]
    print(f"{author}: {[(cats[j], round(R_pred[i, j], 2)) for j in top2]}")

Agatha Christie: [('Juvenile Fiction', np.float64(3.97)), ('Comics & Graphic Novels', np.float64(3.27))]
Charles Dickens: [('Comics & Graphic Novels', np.float64(3.09)), ('Juvenile Nonfiction', np.float64(2.97))]
Clive Cussler: [('Juvenile Fiction', np.float64(4.0)), ('Biography & Autobiography', np.float64(3.38))]
Jane Austen: [('Juvenile Fiction', np.float64(4.14)), ('Comics & Graphic Novels', np.float64(3.62))]
Janet Evanovich: [('Juvenile Fiction', np.float64(3.94)), ('Comics & Graphic Novels', np.float64(3.01))]
John Ronald Reuel Tolkien: [('History', np.float64(4.38)), ('Philosophy', np.float64(3.67))]
Margaret Weis: [('Drama', np.float64(4.02)), ('Juvenile Nonfiction', np.float64(3.99))]
Mercedes Lackey: [('Biography & Autobiography', np.float64(3.95)), ('Juvenile Fiction', np.float64(3.93))]
Neil Gaiman: [('Juvenile Nonfiction', np.float64(3.97)), ('Drama', np.float64(3.96))]
Orson Scott Card: [('Juvenile Fiction', np.float64(3.72)), ('Comics & Graphic Novels', np.float64(2.98)